In [ ]:
!mkdir /content/dataset/

In [ ]:
import json
with open("/content/dataset/data.json", 'r') as f:
  full_data= json.load(f)
words_dict= full_data['word_counts']
words_list=list(words_dict.keys())
words_list = sorted(words_list, key=lambda item: (len(item), item), reverse=True)
print(words_list)

['responsibility', 'administration', 'understanding', 'international', 'circumstances', 'approximately', 'requirements', 'relationship', 'professional', 'particularly', 'organization', 'nevertheless', 'institutions', 'distribution', 'construction', 'considerable', 'traditional', 'temperature', 'significant', 'responsible', 'possibility', 'performance', 'opportunity', 'interesting', 'information', 'individuals', 'independent', 'immediately', 'established', 'educational', 'differences', 'development', 'corporation', 'association', 'washington', 'vocational', 'university', 'understand', 'throughout', 'themselves', 'techniques', 'successful', 'scientific', 'remembered', 'relatively', 'recognized', 'providence', 'production', 'principles', 'population', 'philosophy', 'particular', 'operations', 'membership', 'management', 'literature', 'leadership', 'interested', 'industrial', 'individual', 'increasing', 'impossible', 'importance', 'historical', 'government', 'frequently', 'facilities', 'ex

In [ ]:
test_cases= full_data['test_cases']
cases_list= [ test_case['input'] for test_case in test_cases]
ground_truth= [ test_case['ground_truth'] for test_case in test_cases]


In [ ]:
def greedy_segmenter(words_list, cases_list):
  results = []
  for case in cases_list:
    segmented = []
    i = 0
    while i < len(case):
      match_found = False
      for word in words_list:
        if case.startswith(word, i):
          segmented.append(word)
          i += len(word)
          match_found = True
          break

      if not match_found:
        segmented.append(case[i])
        i += 1
    results.append(' '.join(segmented))
  return results

In [ ]:
import math

def dp_segmenter(words_dict, cases_list):
    total_words = sum(words_dict.values())
    log_probs = {w: math.log(c / total_words) for w, c in words_dict.items()}
    penalty = math.log(1 / (total_words * 100))

    results = []
    for case in cases_list:
        n = len(case)
        dp = [-float('inf')] * (n + 1)
        parent = [0] * (n + 1)

        dp[0] = 0

        for i in range(1, n + 1):
            for j in range(i):
                word = case[j:i]
                if word in log_probs:
                    prob = dp[j] + log_probs[word]
                else:
                    prob = dp[j] + (penalty * (i - j))

                if prob > dp[i]:
                    dp[i] = prob
                    parent[i] = j

        segmented = []
        curr = n
        while curr > 0:
            prev = parent[curr]
            segmented.append(case[prev:curr])
            curr = prev

        results.append(" ".join(reversed(segmented)))

    return results

In [ ]:
def edit_distance(word1, word2):
  m= len(word1)
  n= len(word2)
  dp=[[0 for j in range(n+1)]for i in range(m+1)]
  for i in range(1, m+1):
    dp[i][0]= i;
  for j in range(n+1):
    dp[0][j]= j;

  for i in range(1, m+1):
    for j in range(1, n+1):
      if(word1[i-1]==word2[j-1]):
        dp[i][j]= dp[i-1][j-1]
      else:
        dp[i][j]= min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])+1
  return dp[m][n]

def find_avg_edit_distance(results, ground_truths):
  count = len(results)
  dist= 0
  for i in range(count):
    dist+=edit_distance(results[i], ground_truths[i])
  return dist/count


def find_avg_accuracy(results, ground_truths):
  count = len(results)
  acc= 0
  for i in range(count):
    if(results[i]==ground_truths[i]):
      acc+=1
  return acc/count


In [ ]:
greedy_results = greedy_segmenter(words_list, cases_list)
for i in range(5):
  print(greedy_results[i] + "\n" + ground_truth[i]+"\n")
greedy_edit_distance = find_avg_edit_distance(greedy_results, ground_truth)
print("\n\n")
print(f"Greedy Segmenter Edit Distance: {greedy_edit_distance:.4f}")
greedy_accuracy = find_avg_accuracy(greedy_results, ground_truth)
print(f"Greedy Segmenter Accuracy: {greedy_accuracy:.4f}")

it that the city takes t e p s to this problem
it that the city take steps to this problem

of title law was also by the
of title law was also by the

failure to do this will continue top l a c e a on
failure to do this will continue to place a on

on other the that
on other the that

william for from his wife in court
william for from his wife in court




Greedy Segmenter Edit Distance: 1.2900
Greedy Segmenter Accuracy: 0.6910


In [ ]:
dp_results = dp_segmenter(words_dict, cases_list)
for i in range(5):
  print(dp_results[i] + "\n" + ground_truth[i]+"\n")
greedy_edit_distance = find_avg_edit_distance(dp_results, ground_truth)
dp_edit_distance = find_avg_edit_distance(dp_results, ground_truth)
print(f"DP Segmenter Edit Distance: {dp_edit_distance:.4f}")
dp_accuracy = find_avg_accuracy(dp_results, ground_truth)
print(f"DP Segmenter Accuracy: {dp_accuracy:.4f}")

it that the city take steps to this problem
it that the city take steps to this problem

of title law was also by the
of title law was also by the

failure to do this will continue to place a on
failure to do this will continue to place a on

on other the that
on other the that

william for from his wife in court
william for from his wife in court

DP Segmenter Edit Distance: 0.0280
DP Segmenter Accuracy: 0.9820
